# Dijet reco-to-gen closures

This notebook follows `macro/plotMcClosures.C::plotDiJetClosures`: it projects the configured $p_T^{ave}$ interval, first compares the unnormalized full Reco, Ref, and Gen $\eta_{CM}^{dijet}$ distributions, then unit-normalizes the full distributions with ROOT `TH1::Scale`, and constructs unnormalized forward/backward ratios with ROOT `TH1::Divide`. Each observable has a standalone distribution figure and a separate figure comparing the curves to Gen.

Add another reco-like or smeared distribution by appending one `DijetClosureCurve` below. Supply its full CM histogram-key template; Forward and Backward are folded from the raw full projection.

Input resolution uses reusable files for the requested jet-selection stem, then the original ROOT file, and raises an error if neither exists. Source eta-cut tables must match the current producer; stale inputs require regeneration. `ETA_CUT_VALUE` (where present) is the physical per-jet acceptance, default 1.9; its index is derived from the shared table. `SINGLE_JET_BINNING` and `DIJET_BINNING` independently select existing `normal` or `test` intervals. Existing `PTAVE_BIN_SET` controls use `standard` as the name for normal binning. Trigger-specific normal bins remain the notebook defaults; test dijet bins use MB 50–100, Jet80 100–180, and Jet100 above 180 GeV. Jet60 has no test interval. Diagnostic frame scans and trigger-stitching overlap windows retain their established definitions.

Display comparisons default to ROOT independent-error propagation. Shared-event and normalization correlations are not included. Binomial errors require an actual numerator-subset efficiency and are not appropriate for unfolded, smeared, normalized-shape, or F/B-to-F/B comparisons.

<!-- detailed-workflow-guide -->

### Detailed workflow and closure meaning

The closure ratio in bin $i$ is $C_i=N_i^{reco}/N_i^{gen}$, or the corresponding normalized-shape ratio when normalization is requested. A value near one means the selected reconstruction/smearing model reproduces that generator distribution within the displayed statistical precision. Independent reco and gen weighted histograms generally have correlated origins, although the displayed ROOT ratio uses the configured approximation.

Forward/backward closure is built from unnormalized eta projections as $F/B(|\eta|)=N(+|\eta|)/N(-|\eta|)$ with independent errors. Normalizing the full eta distribution before F/B is unnecessary because a common factor cancels. Empty denominators are undefined and must not be interpreted as physical zero ratios.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.dijet_closure_plots import (
    closure_styles, draw_dijet_comparison, draw_dijet_figure,
    save_dijet_grid, signed_eta_plot_range,
)
from hist_analysis.python.plot_labels import beam_orientation_label, CMS_SIMULATION_HEADER, PPB_HEADER_RIGHT
from hist_analysis.python.root_style import (
    configure_cms_header,
)

CMS_HEADER_LEFT = CMS_SIMULATION_HEADER
CMS_HEADER_RIGHT = PPB_HEADER_RIGHT
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)

from hist_analysis.python.root_style import configure_mplhep_export

SAVE_MPLHEP = True  # retain ROOT output and optionally add mplhep PDFs
SAVE_MPLHEP_PNG = False
configure_mplhep_export(SAVE_MPLHEP, save_png=SAVE_MPLHEP_PNG)


In [ ]:
# Cell role: perform analysis step 2.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration

The defaults compare nominal Gen, Reco, and matched Ref and select the standard 1.9 jet-eta cut by its canonical macro index. Set `ETA_CUT_INDICES = range(len(ETA_CUTS))` to process all eight cuts. The raw full-distribution closure is always built with `normalization='none'` before the separately configured normalized-shape closure. `NORMALIZATION = 'integral'` gives a per-bin unit-area shape; `bin_width` produces a unit-area density. `FORWARD_BACKWARD_RATIO_OPTION` must remain `''` because Forward and Backward are independent weighted populations. The later ratio of one already-constructed F/B histogram to another is configured separately with `FB_CLOSURE_RATIO_OPTION`; the user may select `''` or `'B'`.

In [ ]:
DIJET_BINNING = 'normal'  # normal or test
ETA_CUT_VALUE = 1.9  # per-jet |etaCM| <= value; supported producer cuts only
from hist_analysis.config.histograms import pt_bins, eta_cut_index
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
from hist_analysis.config.histograms import DIJET_ETA_CUTS, STANDARD_DIJET_ETA_CUT_INDEX
ETA_CUTS = DIJET_ETA_CUTS  # shared current production cut order
ETA_CUT_INDICES = (eta_cut_index(ETA_CUT_VALUE),)  # 1.9
REBIN_ETA = 2
NORMALIZATION = 'integral'   # macro default: none, integral, or bin_width
FORWARD_BACKWARD_RATIO_OPTION = ''  # F / B: never use binomial errors
FULL_CLOSURE_RATIO_OPTION = 'B'  # independent display errors; shared-event covariance is not included
FB_CLOSURE_RATIO_OPTION = ''  # independent display errors; shared-event covariance is not included
FULL_RATIO_RANGE = (0.75, 1.25)
RAW_FULL_RATIO_RANGE = (0.75, 1.25)
FB_RATIO_RANGE = (0.75, 1.3)
FB_DOUBLE_RATIO_RANGE = (0.75, 1.25)
FB_Y_TITLE = 'Forward (#eta_{CM}^{dijet} > 0) / Backward (#eta_{CM}^{dijet} < 0)'
SAVE_PNG = False
DRAW_GRID = True
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_RECO_TO_GEN_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'dijet_reco_to_gen_closures',
))

CURVES = (
    DijetClosureCurve(
        'Reco', 'hRecoDijetPtEtaCM_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'Gen', 'hGenDijetPtEtaCM_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'Ref', 'hRefDijetPtEtaCM_{eta_cut_index}'),
    # DijetClosureCurve(
    #     'RefSel', 'hRefSelDijetPtEtaCM_{eta_cut_index}',
    # ),
    # DijetClosureCurve('Reco JER x1.0', 'hRecoDijetPtEtaCMJerDef_{eta_cut_index}'),
)
PLOT_STYLES = closure_styles(curve.label for curve in CURVES)
DIJET_PTAVE_BINS = pt_bins('dijet', DIJET_BINNING)

# Output identity uses the actual configured selections. Each execution gets
# dated renderer directories and deterministic filenames within the date.
from hist_analysis.python.output_naming import OutputDirectory, OutputSelection, nominal_jet_eta_cut
OUTPUT_SELECTIONS = OutputSelection(FILE_STEM, [ETA_CUTS[i] for i in ETA_CUT_INDICES], DIJET_PTAVE_BINS, 'ptAve')
OUTPUT_DIR = OutputDirectory(OUTPUT_DIR, OUTPUT_SELECTIONS, workflow_root=Path(os.environ.get('DIJET_RECO_TO_GEN_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis' / 'output' / 'dijet_reco_to_gen_closures')))

# One deterministic ROOT archive; rerunning configuration starts a fresh file.
from hist_analysis.python.notebook_output import NotebookRootOutput
ROOT_OUTPUT = NotebookRootOutput(OUTPUT_DIR, '03_mc_dijet_reco_to_gen_closures')


In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

if DIRECTION not in {'pgoing', 'Pbgoing', 'combined'}:
    raise ValueError(f'Unsupported DIRECTION={DIRECTION!r}')
DATASET_LABEL = ', '.join(filter(None, (GENERATOR.capitalize(), beam_orientation_label(DIRECTION))))

INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')
INPUT_FILE

## Build projections and ROOT ratios

The raw full distributions are projected and compared before any normalization. A second set of full distributions is then normalized according to `NORMALIZATION`. Forward and backward projections are deliberately not normalized before division. With the default `integral` mode, each normalized full eta shape has a unit sum of in-range bin contents; `bin_width` instead makes the bin-width-weighted integral equal to one.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
closure_results = {}

for eta_cut_index in ETA_CUT_INDICES:
    if eta_cut_index < 0 or eta_cut_index >= len(ETA_CUTS):
        raise IndexError(f'Invalid eta-cut index: {eta_cut_index}')
    eta_cut = ETA_CUTS[eta_cut_index]
    fb_x_range = (0.0, eta_cut + 0.1)
    for ptave_range in DIJET_PTAVE_BINS:
        raw_eta_shapes, _, raw_keys = build_dijet_gen_comparisons(
            INPUT_FILE, CURVES, eta_cut_index=eta_cut_index,
            ptave_range=ptave_range, nominal='Gen',
            rebin_eta=REBIN_ETA, normalization='none',
            ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
        )
        eta_shapes, fb_ratios, keys = build_dijet_gen_comparisons(
            INPUT_FILE, CURVES, eta_cut_index=eta_cut_index,
            ptave_range=ptave_range, nominal='Gen',
            rebin_eta=REBIN_ETA, normalization=NORMALIZATION,
            ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
        )
        eta_cut_tag = int(round(10.0 * eta_cut))
        ptave_tag = f'{ptave_range[0]:g}_{ptave_range[1]:g}'.replace('.', 'p')
        selection_tag = f'{GENERATOR}_{DIRECTION}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        full_output_tag = (
            f'{GENERATOR}_{DIRECTION}_full_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        )
        raw_full_output_tag = (
            f'{GENERATOR}_{DIRECTION}_full_raw_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        )
        fb_output_tag = (
            f'{GENERATOR}_{DIRECTION}_fb_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        )
        annotations = (
            DATASET_LABEL,
            f'Reco jet selection: {FILE_STEM}',
            f'{ptave_range[0]:g} #leq p_{{T}}^{{ave}} < {ptave_range[1]:g} GeV',
            f'|#eta_{{CM}}^{{jet}}| #leq {eta_cut:g}',
            'p_{T}^{Lead} > 50 GeV, p_{T}^{SubLead} > 40 GeV',
            DIJET_DELTA_PHI_SELECTION_LABEL,
        )
        figure_options = dict(
            output_dir=OUTPUT_DIR, styles=PLOT_STYLES,
            grid=DRAW_GRID, save_png=SAVE_PNG,
        )
        eta_y_title = {
            'none': 'dN/d#eta_{CM}^{dijet}',
            'integral': '1/N dN/d#eta_{CM}^{dijet}',
            'bin_width': '1/N dN/d#eta_{CM}^{dijet}',
        }[NORMALIZATION]
        raw_eta_canvas, raw_eta_ratio_canvas, raw_eta_to_gen = draw_dijet_comparison(
            raw_eta_shapes, 'Gen', output_tag=raw_full_output_tag,
            y_title='dN/d#eta_{CM}^{dijet}', ratio_y_title='Ratio to Gen',
            ratio_range=RAW_FULL_RATIO_RANGE, ratio_option=FULL_CLOSURE_RATIO_OPTION,
            x_range=signed_eta_plot_range(raw_eta_shapes['Gen'], eta_cut),
            annotations=annotations, **figure_options,
        )
        eta_canvas, eta_ratio_canvas, eta_to_gen = draw_dijet_comparison(
            eta_shapes, 'Gen', output_tag=full_output_tag,
            y_title=eta_y_title, ratio_y_title='Ratio to Gen',
            ratio_range=FULL_RATIO_RANGE, ratio_option=FULL_CLOSURE_RATIO_OPTION,
            x_range=signed_eta_plot_range(eta_shapes['Gen'], eta_cut),
            annotations=annotations, **figure_options,
        )
        fb_canvas, fb_ratio_canvas, fb_to_gen = draw_dijet_comparison(
            fb_ratios, 'Gen', output_tag=fb_output_tag,
            y_title=FB_Y_TITLE, ratio_y_title='(F/B) / (F/B)_{Gen}',
            ratio_range=FB_DOUBLE_RATIO_RANGE, ratio_option=FB_CLOSURE_RATIO_OPTION,
            annotations=annotations, x_range=fb_x_range,
            y_range=FB_RATIO_RANGE, fb=True, **figure_options,
        )
        closure_results[selection_tag] = {
            'raw_eta_shapes': raw_eta_shapes,
            'raw_eta_to_gen': raw_eta_to_gen,
            'eta_shapes': eta_shapes, 'eta_to_gen': eta_to_gen,
            'forward_backward': fb_ratios, 'forward_backward_to_gen': fb_to_gen,
            'raw_eta_canvas': raw_eta_canvas, 'raw_eta_ratio_canvas': raw_eta_ratio_canvas,
            'eta_canvas': eta_canvas, 'eta_ratio_canvas': eta_ratio_canvas,
            'forward_backward_canvas': fb_canvas, 'forward_backward_ratio_canvas': fb_ratio_canvas,
            'raw_keys': raw_keys, 'keys': keys,
        }
        print(selection_tag, keys)
        display(raw_eta_canvas, raw_eta_ratio_canvas, eta_canvas, eta_ratio_canvas,
                fb_canvas, fb_ratio_canvas)

In [ ]:
# Save four 4×4 summaries in the configured pTave order.
grid_ptave_ranges = DIJET_PTAVE_BINS[:16]
if len(grid_ptave_ranges) == 16:
    grid_categories = {
        'full_distributions': 'eta_canvas',
        'full_ratios_to_gen': 'eta_ratio_canvas',
        'fb_distributions': 'forward_backward_canvas',
        'fb_ratios_to_gen': 'forward_backward_ratio_canvas',
    }
    for eta_cut_index in ETA_CUT_INDICES:
        eta_cut = ETA_CUTS[eta_cut_index]
        eta_cut_tag = int(round(10.0 * eta_cut))
        grid_selection = OutputSelection(FILE_STEM, [eta_cut], grid_ptave_ranges, 'ptAve')
        grid_results = []
        for low, high in grid_ptave_ranges:
            ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
            key = f'{GENERATOR}_{DIRECTION}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
            grid_results.append(closure_results[key])
        for category, canvas_key in grid_categories.items():
            grid_name = f'{GENERATOR}_{DIRECTION}_{category}_etaCM_{eta_cut_tag}_grid4x4.pdf'
            save_dijet_grid(
                [result[canvas_key] for result in grid_results],
                output=OUTPUT_DIR.file(grid_name, selection=grid_selection),
                cms_header_left=CMS_HEADER_LEFT, cms_header_right=CMS_HEADER_RIGHT,
                save_png=SAVE_PNG,
            )
else:
    print('4×4 summaries require at least 16 configured pTave intervals.')


In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
# For each configured curve and pTave interval, compare F/B across eta cuts.
forward_backward_eta_cut_overlay_results = {}
if not ETA_CUT_INDICES:
    raise ValueError('ETA_CUT_INDICES must contain at least one eta-cut index')
invalid_eta_cut_indices = [
    index for index in ETA_CUT_INDICES if index < 0 or index >= len(ETA_CUTS)
]
if invalid_eta_cut_indices:
    raise IndexError(f'Invalid eta-cut indices: {invalid_eta_cut_indices}')
overlay_fb_x_range = (
    0.0, max(ETA_CUTS[index] for index in ETA_CUT_INDICES) + 0.1,
)

for curve in CURVES:
    curve_tag = ''.join(character.lower() if character.isalnum() else '_'
                        for character in curve.label).strip('_')
    for ptave_range in DIJET_PTAVE_BINS:
        ptave_tag = f'{ptave_range[0]:g}_{ptave_range[1]:g}'.replace('.', 'p')
        output_tag = (
            f'{GENERATOR}_{DIRECTION}_{curve_tag}_fb_etaCutOverlay_ptave_{ptave_tag}'
        )
        overlay_histograms = {}
        cut_curves = {}
        for eta_cut_index in ETA_CUT_INDICES:
            eta_cut = ETA_CUTS[eta_cut_index]
            eta_cut_tag = int(round(10.0 * eta_cut))
            selection_tag = (
                f'{GENERATOR}_{DIRECTION}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
            )
            source_histogram = closure_results[selection_tag]['forward_backward'][curve.label]
            histogram = source_histogram.Clone(f'h_{output_tag}_etaCM_{eta_cut_tag}')
            histogram.SetDirectory(0)
            overlay_histograms[eta_cut_index] = histogram
            cut_curves[f'|#eta_{{CM}}^{{jet}}| #leq {eta_cut:g}'] = histogram

        canvas = draw_dijet_figure(
            cut_curves, output_dir=OUTPUT_DIR, output_tag=output_tag,
            x_title='|#eta_{CM}^{dijet}|', y_title=FB_Y_TITLE,
            x_range=overlay_fb_x_range, y_range=FB_RATIO_RANGE,
            styles=dict.fromkeys(cut_curves, PLOT_STYLES[curve.label]),
            line_styles={label: index for index, label in enumerate(cut_curves, 1)}
                        if len(cut_curves) > 1 else None,
            grid=DRAW_GRID, save_png=SAVE_PNG,
            annotations=(
                DATASET_LABEL,
                f'{curve.label} dijets, reco selection: {FILE_STEM}',
                f'{ptave_range[0]:g} #leq p_{{T}}^{{ave}} < {ptave_range[1]:g} GeV',
                'p_{T}^{Lead} > 50 GeV, p_{T}^{SubLead} > 40 GeV',
                DIJET_DELTA_PHI_SELECTION_LABEL,
            ),
        )
        result_key = (curve.label, ptave_range)
        forward_backward_eta_cut_overlay_results[result_key] = {
            'canvas': canvas, 'histograms': overlay_histograms,
        }
        display(canvas)

## Inspect numerical results

The retained dictionaries contain the raw and normalized ROOT histograms, closure ratios, and canvases for interactive inspection. This cell reports each raw yield, the normalized integrals, and the extrema of the raw-full, normalized-full, and F/B closure ratios to Gen.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
for tag, result in closure_results.items():
    print(f'\n{tag}')
    for label in (curve.label for curve in CURVES if curve.label != 'Gen'):
        raw_eta_ratio = result['raw_eta_to_gen'][label]
        eta_ratio = result['eta_to_gen'][label]
        fb_ratio = result['forward_backward_to_gen'][label]
        eta_values = [eta_ratio.GetBinContent(i) for i in range(1, eta_ratio.GetNbinsX() + 1)
                      if eta_ratio.GetBinContent(i) != 0.0]
        raw_eta_values = [
            raw_eta_ratio.GetBinContent(i)
            for i in range(1, raw_eta_ratio.GetNbinsX() + 1)
            if raw_eta_ratio.GetBinContent(i) != 0.0
        ]
        fb_values = [fb_ratio.GetBinContent(i) for i in range(1, fb_ratio.GetNbinsX() + 1)
                     if fb_ratio.GetBinContent(i) != 0.0]
        print(label, {
            'raw_yield': result['raw_eta_shapes'][label].Integral(),
            'raw/gen range': (min(raw_eta_values), max(raw_eta_values))
                             if raw_eta_values else None,
            'bin_sum': result['eta_shapes'][label].Integral(),
            'width_integral': result['eta_shapes'][label].Integral('width'),
            'eta/gen range': (min(eta_values), max(eta_values)) if eta_values else None,
            '(F/B)/(F/B)_gen range': (min(fb_values), max(fb_values)) if fb_values else None,
        })

### ROOT histogram archive
The file below includes plot-time histogram snapshots and retained results, with configuration metadata. Rerunning the configuration cell recreates the same file; rerunning plot cells replaces their keys.

In [ ]:
# Save retained numerical products in addition to exact plot-time snapshots.
NOTEBOOK_ROOT_FILE = ROOT_OUTPUT.finish(globals())
